# 🧭 Step 4 · Build the Similarity-Search Index

**Embedding pipeline.** Encode every face image with CLIP and store the vectors in an online index for k-NN search.

`images → 🧠 CLIP → vectors → 🗄️ image_embeddings (vector index)`

In [ ]:
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import hopsworks
import os

proj = hopsworks.login()
fs = proj.get_feature_store()
mr = proj.get_model_registry()

### 🧠 Load the CLIP model
Fetch the embedding model from the registry (or download from Hugging Face on first run).

In [ ]:
name = "openaiclip_vit_base_patch32"
model_mr = mr.get_model(name, version=1)

if model_mr is None:
    print("Model not in registry — downloading from Hugging Face")
    save_dir = "/tmp/clip-vit-base-patch32-local"
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)
    model_mr = mr.python.create_model(name=name, description="Image vector embedding model from OpenAI")
    model_mr.save(save_dir)
    print(f"✅ Registered model '{name}' v{model_mr.version}")
else:
    print("Downloading model from the model registry")
    local_path = model_mr.download()
    model = CLIPModel.from_pretrained(local_path)
    processor = CLIPProcessor.from_pretrained(local_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(f"✅ Model ready on device: {device}")

### 🔢 Embedding function
Encode an image into a normalized CLIP vector.

In [ ]:
def _embed_pil(images):
    """Core: list of PIL images -> list of L2-normalized CLIP vectors."""
    inputs = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.get_image_features(**inputs)
    feats = getattr(out, "pooler_output", out)              # transformers 5.x: .pooler_output; 4.x: tensor
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)    # L2-normalize
    return feats.cpu().tolist()


def embed_image(image):
    """Embed a single image (PIL.Image or file path) -> one vector."""
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")
    return _embed_pil([image])[0]


def embed_paths(paths, batch_size=64):
    """Embed many image paths in GPU batches -> list of vectors (None for unreadable files)."""
    vectors = [None] * len(paths)
    total = len(paths)
    for start in range(0, total, batch_size):
        batch_idx, batch_img = [], []
        for j in range(start, min(start + batch_size, total)):
            try:
                batch_img.append(Image.open(paths[j]).convert("RGB"))
                batch_idx.append(j)
            except Exception as e:
                print(f"  ⚠️  skipping {paths[j]}: {e}")
        for j, vec in zip(batch_idx, _embed_pil(batch_img)):
            vectors[j] = vec
        print(f"  embedded {min(start + batch_size, total)}/{total} images")
    return vectors

### 📥 Read the image records

In [ ]:
fg =  fs.get_feature_group("wider_face_files", version=1)
df = fg.read()
df

In [ ]:
df1 = df[["file_path"]].copy()
df1.head()

### ⚡ Compute embeddings

In [ ]:
import time

start = time.time()
df1["embedding"] = embed_paths(df1["file_path"].tolist(), batch_size=64)
print(f"✅ Embedded {len(df1)} images in {time.time() - start:.1f}s")
df1.head()

### 🗄️ Create the vector-index feature group
Store embeddings in the online **`image_embeddings`** group with a k-NN index.

In [ ]:
from hopsworks import hsfs
from hsfs.statistics_config import StatisticsConfig

embedding_index = hsfs.embedding.EmbeddingIndex()
embedding_index.add_embedding("embedding", dimension=model.config.projection_dim, model=model_mr)

# Create or get feature group. kll=True persists per-commit KLL sketches, required by the
# rolling-window distribution monitoring configured in notebook 6.
fg = fs.get_or_create_feature_group(
    name="image_embeddings",
    version=1,
    primary_key=["file_path"],
    online_enabled=True,
    description="image embeddings",
    embedding_index=embedding_index,
    statistics_config=StatisticsConfig(kll=True),
)
fg.insert(df1)

## 🔎 Test similarity search
Embed a query image and retrieve the nearest faces.

In [ ]:
image="./data/images/bus.jpg"

img = Image.open(image)
display(img)

In [ ]:
query_vector = embed_image(image)
print(f"Query embedding dim: {len(query_vector)}")

results = fg.find_neighbors(query_vector, k=5)
print(f"Top {len(results)} matches:")
for result in results:
    path = result[1][0]      # result = (distance, [feature values]); file_path is first
    print(" ", path)
    display(Image.open(path))

In [ ]:
# Direct access to OpenSearch API


# from opensearchpy import OpenSearch, exceptions

# print(fgSS.embedding_index)
# opensearch_api = proj.get_opensearch_api()
# client = OpenSearch(**opensearch_api.get_default_py_config())
# index_name = "120__embedding_default_project_embedding_0"
# try:
#     res = client.search(index=index_name, size=1)
#     print("✅ Read access works")
#     if "hits" in res:
#         print("Got", res["hits"]["total"], "documents")
# except exceptions.NotFoundError:
#     print("❌ Index does not exist")
# except exceptions.AuthorizationException:
#     print("❌ You don’t have read permissions for this index")
# except Exception as e:
#     print("⚠️ Other error:", e)
